In [7]:
# ===== Import libraries =====
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score


# ===== Load dataset =====
url = "https://drive.google.com/file/d/1JPFEjSOlQ-gWExGiUv2kB1-yHWGfZdOQ/view?usp=sharing"
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]

housing = pd.read_csv(path)
housing.head()

# ===== Define features and target =====
# Remove Id because it is only an identifier, not a real feature
X = housing.drop(columns=["SalePrice", "Id"], errors="ignore")
y = np.log1p(housing["SalePrice"])   # log target for Kaggle metric

# ===== Split data =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===== Separate categorical and numerical features =====
X_cat = X_train.select_dtypes(include=["object"])
X_num = X_train.select_dtypes(exclude=["object"])

# ===== Preprocessing =====
preprocessor = make_column_transformer(
    (
        make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        ),
        X_cat.columns
    ),
    (
        SimpleImputer(strategy="median"),
        X_num.columns
    ),
    remainder="drop"
)

# ===== Build pipeline =====
# DecisionTree is used only for feature selection
# SGDRegressor is the final prediction model
sfm_sgd_pipe = make_pipeline(
    preprocessor,
    StandardScaler(),
    SelectFromModel(
        estimator=DecisionTreeRegressor(random_state=42),
        threshold="median"
    ),
    SGDRegressor(random_state=42, max_iter=2000, tol=1e-3)
)

# ===== Cross-validation with RMSE on log scale =====
cv_scores = cross_val_score(
    sfm_sgd_pipe,
    X_train,
    y_train,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

cv_rmse = -cv_scores

print("CV RMSE (log scale):", cv_rmse)
print("Mean CV RMSE (log scale):", cv_rmse.mean())
print("Std CV RMSE:", cv_rmse.std())

# ===== Train model =====
sfm_sgd_pipe.fit(X_train, y_train)

# ===== Predict on internal test set =====
y_pred_log = sfm_sgd_pipe.predict(X_test)

# ===== Evaluation on log scale =====
print("\n--- LOG SCALE ---")
print("Test MAE:", mean_absolute_error(y_test, y_pred_log))
print("Test RMSE:", mean_squared_error(y_test, y_pred_log) ** 0.5)
print("Test MAPE:", mean_absolute_percentage_error(y_test, y_pred_log))
print("Test R2:", r2_score(y_test, y_pred_log))

# ===== Convert back to original SalePrice scale =====
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log)

# ===== Evaluation on original SalePrice scale =====
print("\n--- ORIGINAL PRICE SCALE ---")
print("Test RMSE:", mean_squared_error(y_test_real, y_pred_real) ** 0.5)

# ===== Show selected feature names =====
feature_names = sfm_sgd_pipe.named_steps["columntransformer"].get_feature_names_out()
selected_mask = sfm_sgd_pipe.named_steps["selectfrommodel"].get_support()
selected_features = pd.Series(feature_names[selected_mask], name="Selected Features")

print("\nSelected features:")
print(selected_features.to_string(index=False))

CV RMSE (log scale): [0.13232311 0.17643599 0.2231342  0.13731911 0.12917128]
Mean CV RMSE (log scale): 0.15967673786584785
Std CV RMSE: 0.036017744505761756

--- LOG SCALE ---
Test MAE: 0.11744842679934277
Test RMSE: 0.1588956381388685
Test MAPE: 0.009893287492035574
Test R2: 0.8647034453305378

--- ORIGINAL PRICE SCALE ---
Test RMSE: 29243.849116765636

Selected features:
       pipeline__LandContour
      pipeline__Neighborhood
         pipeline__RoofStyle
       pipeline__Exterior1st
         pipeline__ExterQual
        pipeline__Foundation
          pipeline__BsmtQual
      pipeline__BsmtExposure
      pipeline__BsmtFinType1
      pipeline__BsmtFinType2
        pipeline__CentralAir
       pipeline__KitchenQual
        pipeline__Functional
        pipeline__GarageType
      pipeline__GarageFinish
             pipeline__Fence
  simpleimputer__LotFrontage
      simpleimputer__LotArea
  simpleimputer__OverallQual
  simpleimputer__OverallCond
    simpleimputer__YearBuilt
 simpleimputer

#COMPETITION-Kaggle

In [8]:
# ===== Load teacher's external test data =====
url = "https://drive.google.com/file/d/1q14sdW_8Gk9x0h5fAejPpq38xuRwgHnY/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id=" + url.split("/")[-2]

# ===== Load teacher test CSV =====

test_df = pd.read_csv(path)

In [10]:
# ===== Use test data WITHOUT dropping Id =====
X_teacher_test = test_df.copy()

# ===== Predict =====
test_pred_log = sfm_sgd_pipe.predict(X_teacher_test)

# ===== Convert back =====
test_pred = np.expm1(test_pred_log)

# ===== Build submission =====
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_pred
})

# ===== Save =====
submission.to_csv("submission_SFM_SGD_R.csv", index=False)

submission.head()

,Id,SalePrice
0,1461,116930.612391
1,1462,151983.745578
2,1463,172598.853798
3,1464,191363.543102
4,1465,206307.666698


In [11]:
from google.colab import files
files.download("submission_SFM_SGD_R.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>